In [19]:
#NCSG Team 5 Data Dashboard 4/24/26 updated

#Imports and Configs

from pathlib import Path
import requests
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Add your FRED API key here
FRED_API_KEY = "9e5d944924918a26660adf4a492e447e"  # PLEASE REPLACE WITH YOUR OWN KEY

# Output folders
OUTPUT_DIR = Path("plotly_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# Maryland economic series (FRED)
SERIES_CONFIG = {
    "real_gdp": ("MDRGSP", "Real GDP", "Millions of Dollars", "line"),
    "population": ("MDPOP", "Resident Population", "Thousands", "line"),
    "income": ("MEHOINUSMDA672N", "Real Median Income", "Dollars", "line"),
    "poverty": ("PPAAMD24000A156NCEN", "Poverty Rate", "%", "line"),
    "business": ("BABATOTALSAMD", "Business Applications (EIN)", "Applications", "bar")
}

# Maryland foreclosure API
FORECLOSURE_API = "https://opendata.maryland.gov/resource/w3bc-8mnv.json"

In [14]:
#originally the past group had this as cited from BLS, but I used fred to do get the economic data

def fetch_fred(series_id):
    url = "https://api.stlouisfed.org/fred/series/observations"
    params = {
        "series_id": series_id,
        "api_key": FRED_API_KEY,
        "file_type": "json"
    }

    #get requests and convert to dataframe
    res = requests.get(url, params=params)
    data = res.json()

    df = pd.DataFrame(data["observations"])
    df = df[df["value"] != "."]
    df["date"] = pd.to_datetime(df["date"])
    df["value"] = pd.to_numeric(df["value"])

    return df[["date", "value"]]

In [15]:
# Load all economic data into a single DataFrame
def load_economic_data():
    frames = []

    # Loop through each series in the config, fetch data, and append to frames
    for key, (sid, label, units, chart) in SERIES_CONFIG.items():
        df = fetch_fred(sid)
        df["metric"] = label
        df["units"] = units
        df["chart"] = chart
        frames.append(df)

    return pd.concat(frames)

economic_df = load_economic_data()
economic_df.head()

,date,value,metric,units,chart
0,1997-01-01,237589.2,Real GDP,Millions of Dollars,line
1,1998-01-01,248662.9,Real GDP,Millions of Dollars,line
2,1999-01-01,258913.4,Real GDP,Millions of Dollars,line
3,2000-01-01,269301.2,Real GDP,Millions of Dollars,line
4,2001-01-01,281053.6,Real GDP,Millions of Dollars,line


In [7]:
def load_foreclosures():
    res = requests.get(FORECLOSURE_API) #different API than FRED, but same process of converting to dataframe and cleaning data
    df = pd.DataFrame(res.json())

    df["date"] = pd.to_datetime(df["date"])

    # get all county columns
    ignore = {"date", "type"}
    counties = [c for c in df.columns if c not in ignore]

    for c in counties:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

    df["total"] = df[counties].sum(axis=1)

    return df[["date", "type", "total"]]

foreclosure_df = load_foreclosures()
foreclosure_df.head()

,date,type,total
0,2026-03-01,Notice of Intent to Foreclose,8074.0
1,2026-03-01,Notice of Foreclosure,818.0
2,2026-03-01,Foreclosure Property Registration,155.0
3,2026-02-01,Notice of Intent to Foreclose,8754.0
4,2026-02-01,Notice of Foreclosure,904.0


In [17]:
#CHART GENERATION

# Create subplots for each economic metric
metrics = economic_df["metric"].unique()

fig = make_subplots(
    rows=len(metrics),
    cols=1,
    subplot_titles=metrics,
    vertical_spacing=0.05
)

# Loop through each metric and add the appropriate trace (line or bar) to the subplot
for i, metric in enumerate(metrics, start=1):
    df_m = economic_df[economic_df["metric"] == metric]

    if df_m["chart"].iloc[0] == "bar":
        fig.add_trace(
            go.Bar(x=df_m["date"], y=df_m["value"], name=metric),
            row=i, col=1
        )
    else:
        fig.add_trace(
            go.Scatter(x=df_m["date"], y=df_m["value"], mode="lines"),
            row=i, col=1
        )

    fig.update_yaxes(title_text=df_m["units"].iloc[0], row=i, col=1)

# Add foreclosure data as a line plot in the last subplot
fig.update_layout(
    height=300 * len(metrics),
    title="Maryland Economic Dashboard",
    template="plotly_white"
)

fig.show()

In [18]:
# Individual Chart Generation for website functionality according to client needs
for metric in economic_df["metric"].unique():
    df_m = economic_df[economic_df["metric"] == metric]

    if df_m["chart"].iloc[0] == "bar":
        fig = go.Figure(go.Bar(
            x=df_m["date"],
            y=df_m["value"]
        ))
    else:
        fig = go.Figure(go.Scatter(
            x=df_m["date"],
            y=df_m["value"],
            mode="lines+markers"
        ))

    fig.update_layout(
        title=metric,
        xaxis_title="Date",
        yaxis_title=df_m["units"].iloc[0],
        template="plotly_white"
    )

    fig.show()

In [10]:
# Forclosure chart generation for website functionality according to client needs
fig = go.Figure()

types = foreclosure_df["type"].unique()

for t in types:
    df_t = foreclosure_df[foreclosure_df["type"] == t]

    fig.add_trace(go.Scatter(
        x=df_t["date"],
        y=df_t["total"],
        mode="lines",
        name=t
    ))

fig.update_layout(
    title="Maryland Foreclosures",
    xaxis_title="Date",
    yaxis_title="Total",
    template="plotly_white"
)

fig.show()